# Classify MNIST digits with the fruit-fly connectome reservoir

This notebook restores the exact sparse reservoir graph and learned weights saved by `train.py`. It then predicts MNIST test digits and can classify a custom image. The design is informed by [Haltere](https://github.com/skulitom/haltere), but this small MNIST reservoir is not Haltere's trained 30,000-neuron flight model.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from PIL import Image, ImageOps
from torchvision import datasets, transforms

from train import FlyReservoir

CHECKPOINT = Path("runs/malecns/best.pt")
# Sparse matrix multiplication is reliable on CPU/CUDA; sparse MPS support
# varies by PyTorch release, so Apple Silicon defaults to CPU here.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {DEVICE}")

## Load the trained model

If the checkpoint does not exist, first run the training command from `README.md`.

In [ ]:
if not CHECKPOINT.exists():
    raise FileNotFoundError(
        f"No checkpoint at {CHECKPOINT}. Run train.py first; see README.md."
    )

# The checkpoint is produced locally by train.py and includes its sparse graph.
checkpoint = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
state = checkpoint["model"]
saved_args = checkpoint["args"]
graph = state["graph"].coalesce()
model = FlyReservoir(
    graph=graph,
    steps=int(saved_args["steps"]),
    train_input=not bool(saved_args.get("freeze_input", False)),
).to(DEVICE)
model.load_state_dict(state)
model.eval()

graph_source = saved_args.get("graph", "unknown")
print(f"Loaded {CHECKPOINT}")
print(f"Graph source: {graph_source}")
if graph_source == "synthetic":
    print("NOTE: this is the starter/smoke-test graph, not MaleCNS.")
print(f"Reservoir nodes: {graph.shape[0]:,}; edges: {graph._nnz():,}")

## Prediction helper

In [ ]:
@torch.inference_mode()
def predict(images):
    """Return predicted digits and class probabilities for [N, 1, 28, 28]."""
    if images.ndim == 3:
        images = images.unsqueeze(0)
    logits = model(images.to(DEVICE))
    probabilities = logits.softmax(dim=1).cpu()
    return probabilities.argmax(dim=1), probabilities

## Classify examples from the MNIST test set

In [ ]:
mnist_test = datasets.MNIST(
    "data/mnist", train=False, download=True, transform=transforms.ToTensor()
)
indices = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
images = torch.stack([mnist_test[i][0] for i in indices])
labels = torch.tensor([mnist_test[i][1] for i in indices])
predictions, probabilities = predict(images)

fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for ax, image, truth, prediction, confidence in zip(
    axes.flat, images, labels, predictions, probabilities.max(dim=1).values
):
    ax.imshow(image.squeeze(), cmap="gray")
    color = "green" if prediction == truth else "crimson"
    ax.set_title(
        f"pred {prediction.item()} ({confidence:.1%})\ntrue {truth.item()}",
        color=color,
    )
    ax.axis("off")
plt.tight_layout()

## Measure test-set accuracy

This evaluates the complete 10,000-image MNIST test set.

In [ ]:
test_loader = torch.utils.data.DataLoader(mnist_test, batch_size=256)
correct = total = 0
with torch.inference_mode():
    for batch_images, batch_labels in test_loader:
        batch_predictions, _ = predict(batch_images)
        correct += (batch_predictions == batch_labels).sum().item()
        total += batch_labels.numel()
print(f"Test accuracy: {correct / total:.2%} ({correct:,}/{total:,})")

## Classify your own image

Set `IMAGE_PATH` to a PNG or JPEG. The preprocessing assumes a dark digit on a light background, crops its content, preserves aspect ratio, centers it on a 28×28 black canvas, and inverts it to match MNIST. If your image already has a light digit on a dark background, change `INVERT` to `False`.

In [ ]:
IMAGE_PATH = Path("my_digit.png")
INVERT = True

def prepare_digit(path, invert=True):
    image = Image.open(path).convert("L")
    if invert:
        image = ImageOps.invert(image)
    bbox = image.getbbox()
    if bbox is None:
        raise ValueError("The image appears to be blank.")
    image = image.crop(bbox)
    image.thumbnail((20, 20), Image.Resampling.LANCZOS)
    canvas = Image.new("L", (28, 28), 0)
    left = (28 - image.width) // 2
    top = (28 - image.height) // 2
    canvas.paste(image, (left, top))
    return transforms.ToTensor()(canvas), canvas

if IMAGE_PATH.exists():
    digit_tensor, prepared_image = prepare_digit(IMAGE_PATH, INVERT)
    custom_prediction, custom_probabilities = predict(digit_tensor)
    predicted_digit = custom_prediction.item()
    confidence = custom_probabilities[0, predicted_digit].item()
    plt.imshow(prepared_image, cmap="gray")
    plt.title(f"Prediction: {predicted_digit} ({confidence:.1%})")
    plt.axis("off")
    plt.show()
    print("Probabilities:", {i: round(p.item(), 4) for i, p in enumerate(custom_probabilities[0])})
else:
    print(f"Place an image at {IMAGE_PATH.resolve()} or change IMAGE_PATH, then rerun this cell.")